In [400]:
import openpyxl
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
import numpy as np

### DF_INSIGHTS FUNCTION

This function takes a dataframe as parameter to return valuable insights on the dataframe in a table format using `tabulate` 



In [401]:
def df_insights(df):
    insights = [
        ['Count of Columns', df.shape[1]],
        ['Count of Rows', df.shape[0]],
        ['Missing Values', df.isna().sum().sum()]
        ]
    
    print('\n DF SUMMARY \n')
    print(tabulate(insights, headers=['Parameter', 'Value'], tablefmt='fancy_grid'))

    missingValues = df.isna().sum()
    missingValues = missingValues[missingValues > 0] # AI helped me to reason this line.


    missingValues_insights = [
        [eachCol, missingCount] for eachCol, missingCount in missingValues.items()

    ]
    
    if len(missingValues) > 0:
        print('\n MISSING VALUES \n')
        print(tabulate(missingValues_insights, headers=['COLUMN', 'COUNT OF ROWS MISSING VALUES'], tablefmt='fancy_grid'))

    datatypes_insights = [
        [eachCol, datatype] for eachCol, datatype in df.dtypes.items()
    ]

    print('\n DATATYPES \n')
    print(tabulate(datatypes_insights, headers=['COLUMN', 'DTYPE'], tablefmt='fancy_grid'))

Importing Datasets

# UV_EXPOSURE DATASET 

In [402]:


uv_exposure = pd.read_excel('../datasets/uv-county-exposure.xlsx', sheet_name='UV_County_2000-2024', dtype={'COUNTY_FIPS': str})
uv_exposure.rename({'COUNTY NAME': 'COUNTY_NAME', 'STATENAME': 'STATE_NAME'}, axis=1, inplace=True)
uv_exposure['COUNTY_FIPS'] = uv_exposure['COUNTY_FIPS'].str.zfill(5)
uv_exposure = uv_exposure.sort_values(['STATE_NAME', 'COUNTY_NAME'])
uv_exposure




,STATE_NAME,COUNTY_NAME,COUNTY_FIPS,UV_ Wh/m² (2000-2004),UV_ Wh/m² (2005-2009),UV_ Wh/m² (2010-2014),UV_ Wh/m² (2015-2019),UV_ Wh/m² (2020_2024)
2734,Alabama,Autauga,01001,4781.877818,4774.090182,4843.185939,4701.004606,4785.906061
2159,Alabama,Baldwin,01003,4916.409224,4904.892424,4934.852894,4764.348800,4814.548819
565,Alabama,Barbour,01005,4875.885667,4862.169667,4908.160333,4786.408667,4833.696667
974,Alabama,Bibb,01007,4727.518560,4706.299680,4785.613440,4625.073600,4726.646400
2232,Alabama,Blount,01009,4643.034462,4606.273385,4687.476000,4574.497846,4665.236538
...,...,...,...,...,...,...,...,...
1527,Wyoming,Sweetwater,56037,4780.547753,4819.313629,4731.694674,4753.577473,4910.167898
1517,Wyoming,Teton,56039,4294.326031,4282.941613,4162.277026,4303.009508,4529.784895
2234,Wyoming,Uinta,56041,4730.583808,4797.226752,4687.229184,4736.682624,4936.341280
1990,Wyoming,Washakie,56043,4558.186982,4545.855054,4427.304308,4473.671815,4565.537275


In [403]:
df_insights(uv_exposure)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       8 │
├──────────────────┼─────────┤
│ Count of Rows    │    3107 │
├──────────────────┼─────────┤
│ Missing Values   │       0 │
╘══════════════════╧═════════╛

 DATATYPES 

╒═══════════════════════╤═════════╕
│ COLUMN                │ DTYPE   │
╞═══════════════════════╪═════════╡
│ STATE_NAME            │ str     │
├───────────────────────┼─────────┤
│ COUNTY_NAME           │ str     │
├───────────────────────┼─────────┤
│ COUNTY_FIPS           │ str     │
├───────────────────────┼─────────┤
│ UV_ Wh/m² (2000-2004) │ float64 │
├───────────────────────┼─────────┤
│ UV_ Wh/m² (2005-2009) │ float64 │
├───────────────────────┼─────────┤
│ UV_ Wh/m² (2010-2014) │ float64 │
├───────────────────────┼─────────┤
│ UV_ Wh/m² (2015-2019) │ float64 │
├───────────────────────┼─────────┤
│ UV_ Wh/m² (2020_2024) │ float64 │
╘═══════════════════════╧═════════╛


# MELANOMA INCIDENCE WITH COUNTY INFO DATASET

In [404]:


melanoma_incidence = pd.read_csv('../datasets/melanoma-county-incidence.csv', dtype={'FIPS': str}, skiprows=8, skipfooter=35, engine='python')
melanoma_incidence.drop(columns=['Lower 95% Confidence Interval', 
                                 'Upper 95% Confidence Interval', 
                                 'CI*Rank([rank note])', 
                                 'Lower CI (CI*Rank)', 
                                 'Upper CI (CI*Rank)', 
                                 'Recent 5-Year Trend ([trend note]) in Incidence Rates', 
                                 'Lower 95% Confidence Interval.1', 
                                 'Upper 95% Confidence Interval.1',
                                 'Recent Trend'], inplace=True)
melanoma_incidence.rename({'FIPS': 'COUNTY_FIPS',
                           '2023 Rural-Urban Continuum Codes([rural urban note])': 'RURAL_URBAN_NOTE',
                           'Age-Adjusted Incidence Rate([rate note]) - cases per 100,000': 'AGE_RATE_PER100K', 
                           'Average Annual Count': 'AVG_ANNUAL_COUNT'}, axis=1, inplace=True)
melanoma_incidence = melanoma_incidence[melanoma_incidence['COUNTY_FIPS'] != '00000'] # drops the annual agreggate row that has COUNTY_FIPS = '00000'

# SPLIT COUNTY COLUMN INTO COUNTY AND STATE

melanoma_incidence['COUNTY_NAME'] = melanoma_incidence['County'].str.split(', ', expand=True)[0]
melanoma_incidence['STATE_NAME'] = melanoma_incidence['County'].str.split(', ', expand=True)[1]
melanoma_incidence.insert(0, 'COUNTY_NAME', melanoma_incidence.pop('COUNTY_NAME'))
melanoma_incidence.insert(1, 'STATE_NAME', melanoma_incidence.pop('STATE_NAME'))
melanoma_incidence.drop(columns=['County'], inplace=True)

# CLEANING UP COUNTY COLUMN

melanoma_incidence['COUNTY_NAME'] = melanoma_incidence['COUNTY_NAME'].replace(' County', '', regex=True)

melanoma_incidence = melanoma_incidence.sort_values(['COUNTY_FIPS'])
melanoma_incidence


,COUNTY_NAME,STATE_NAME,COUNTY_FIPS,RURAL_URBAN_NOTE,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
1096,Autauga,Alabama(6),01001,Urban,24,16
217,Baldwin,Alabama(6),01003,Urban,37.2,115
1209,Barbour,Alabama(6),01005,Rural,22.8,8
1909,Bibb,Alabama(6),01007,Urban,15.5,4
1569,Blount,Alabama(6),01009,Urban,19.4,14
...,...,...,...,...,...,...
292,Teton,Wyoming(6),56039,Rural,35.2,10
1836,Uinta,Wyoming(6),56041,Rural,16.4,4
3090,Washakie,Wyoming(6),56043,Rural,*,3 or fewer
3108,Weston,Wyoming(6),56045,Rural,*,3 or fewer


In [405]:
melanoma_incidence['AGE_RATE_PER100K'].unique()

<StringArray>
[  '24 ', '37.2 ', '22.8 ', '15.5 ', '19.4 ',    '* ', '17.9 ', '25.2 ',
  '9.1 ', '17.5 ',
 ...
 '68.1 ', '40.5 ',  '7.5 ',  '8.4 ',  '9.7 ', '47.6 ',   '63 ', '27.7 ',
 '35.2 ',  '3.3 ']
Length: 417, dtype: str

In [406]:
melanoma_incidence['AVG_ANNUAL_COUNT'] = (melanoma_incidence['AVG_ANNUAL_COUNT'].replace('3 or fewer', 1.5))
melanoma_incidence['AVG_ANNUAL_COUNT'].unique()

array(['16', '115', '8', '4', '14', 1.5, '5', '33', '6', '10', '7', '3',
       '29', '15', '17', '28', '9', '27', '13', '39', '20', '120', '25',
       '26', '37', '112', '101', '38', '44', '23', '53', '21', '51', '11',
       '56', '34', '19', '1665', '81', '22', '485', '179', '161', '77',
       '24', '111', '47', '36', '50', '366', '97', '420', '106', '178',
       '152', '1397', '40', '221', '122', '59', '66', '1102', '216',
       '667', '393', '355', '1104', '212', '140', '210', '254', '135',
       '151', '83', '283', '119', '58', '276', '49', '76', '142', '104',
       '12', '136', '129', '171', '60', '65', '63', '228', '207', '52',
       '192', '42', '54', '158', '130', '68', '72', '310', '483', '96',
       '88', '256', '286', '117', '73', '423', '141', '143', '476', '257',
       '163', '64', '234', '604', '260', '479', '306', '145', '71', '335',
       '102', '35', '100', '18', '205', '131', '123', '303', '30', '103',
       '48', '32', '233', '70', '193', '61', '31', '92

In [407]:
# '3 or fewer' values were turned into a safe 1.5 value - nan and 'data not available will also become NaN for calculations
melanoma_incidence['AVG_ANNUAL_COUNT'] = (melanoma_incidence['AVG_ANNUAL_COUNT'].replace('3 or fewer', 1.5))
melanoma_incidence

,COUNTY_NAME,STATE_NAME,COUNTY_FIPS,RURAL_URBAN_NOTE,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
1096,Autauga,Alabama(6),01001,Urban,24,16
217,Baldwin,Alabama(6),01003,Urban,37.2,115
1209,Barbour,Alabama(6),01005,Rural,22.8,8
1909,Bibb,Alabama(6),01007,Urban,15.5,4
1569,Blount,Alabama(6),01009,Urban,19.4,14
...,...,...,...,...,...,...
292,Teton,Wyoming(6),56039,Rural,35.2,10
1836,Uinta,Wyoming(6),56041,Rural,16.4,4
3090,Washakie,Wyoming(6),56043,Rural,*,1.5
3108,Weston,Wyoming(6),56045,Rural,*,1.5


In [408]:
df_insights(melanoma_incidence)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       6 │
├──────────────────┼─────────┤
│ Count of Rows    │    3143 │
├──────────────────┼─────────┤
│ Missing Values   │       2 │
╘══════════════════╧═════════╛

 MISSING VALUES 

╒════════════╤════════════════════════════════╕
│ COLUMN     │   COUNT OF ROWS MISSING VALUES │
╞════════════╪════════════════════════════════╡
│ STATE_NAME │                              2 │
╘════════════╧════════════════════════════════╛

 DATATYPES 

╒══════════════════╤═════════╕
│ COLUMN           │ DTYPE   │
╞══════════════════╪═════════╡
│ COUNTY_NAME      │ str     │
├──────────────────┼─────────┤
│ STATE_NAME       │ str     │
├──────────────────┼─────────┤
│ COUNTY_FIPS      │ str     │
├──────────────────┼─────────┤
│ RURAL_URBAN_NOTE │ str     │
├──────────────────┼─────────┤
│ AGE_RATE_PER100K │ str     │
├──────────────────┼─────────┤
│ AVG_ANNUAL_COUNT │ object  │

### Issue observed when normalizing data types:

There were a coupe of rows that had no data available for columns `AGE_RATE_PER100K` and `AVG_ANNUAL_COUNT`.<br>
I did not want to `.drop()` these rows, because I thought values would be skewed.<br>
Instead, used this guidance found <a href="https://stackoverflow.com/questions/36814100/pandas-to-numeric-for-multiple-columns" target="_blank">here</a> to apply `pd.to_numeric()` and then convert selected columns to numeric dtypes.

In [409]:
str_to_nan = ["AGE_RATE_PER100K", "AVG_ANNUAL_COUNT"]
melanoma_incidence[str_to_nan] = melanoma_incidence[str_to_nan].apply(pd.to_numeric, errors="coerce")

melanoma_incidence

,COUNTY_NAME,STATE_NAME,COUNTY_FIPS,RURAL_URBAN_NOTE,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
1096,Autauga,Alabama(6),01001,Urban,24.0,16.0
217,Baldwin,Alabama(6),01003,Urban,37.2,115.0
1209,Barbour,Alabama(6),01005,Rural,22.8,8.0
1909,Bibb,Alabama(6),01007,Urban,15.5,4.0
1569,Blount,Alabama(6),01009,Urban,19.4,14.0
...,...,...,...,...,...,...
292,Teton,Wyoming(6),56039,Rural,35.2,10.0
1836,Uinta,Wyoming(6),56041,Rural,16.4,4.0
3090,Washakie,Wyoming(6),56043,Rural,NaN,1.5
3108,Weston,Wyoming(6),56045,Rural,NaN,1.5


In [410]:
df_insights(melanoma_incidence)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       6 │
├──────────────────┼─────────┤
│ Count of Rows    │    3143 │
├──────────────────┼─────────┤
│ Missing Values   │    1158 │
╘══════════════════╧═════════╛

 MISSING VALUES 

╒══════════════════╤════════════════════════════════╕
│ COLUMN           │   COUNT OF ROWS MISSING VALUES │
╞══════════════════╪════════════════════════════════╡
│ STATE_NAME       │                              2 │
├──────────────────┼────────────────────────────────┤
│ AGE_RATE_PER100K │                            959 │
├──────────────────┼────────────────────────────────┤
│ AVG_ANNUAL_COUNT │                            197 │
╘══════════════════╧════════════════════════════════╛

 DATATYPES 

╒══════════════════╤═════════╕
│ COLUMN           │ DTYPE   │
╞══════════════════╪═════════╡
│ COUNTY_NAME      │ str     │
├──────────────────┼─────────┤
│ STATE_NAME       │ str     │
├

In [412]:
melanoma_incidence[melanoma_incidence['STATE_NAME'].isna()]

,COUNTY_NAME,STATE_NAME,COUNTY_FIPS,RURAL_URBAN_NOTE,AGE_RATE_PER100K,AVG_ANNUAL_COUNT
2146,District of Columbia(6),NaN,11001,Urban,10.0,68.0
2184,Puerto Rico(6),NaN,72001,Urban,3.3,132.0


These are the only two rows that have null values in this dataset. <br>
The `COUNTY_FIPS` code for this row is `72001`.<br>
Let's evaluate if there is a UV exposure measurement for this county in the `uv_exposure` df:<br>


In [413]:
if uv_exposure[uv_exposure['COUNTY_FIPS'] == '11001'].empty:
    print('FIPS CODE NOT FOUND')
else:
    print('UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE')

uv_exposure[uv_exposure['COUNTY_FIPS'] == '11001']

UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE


,STATE_NAME,COUNTY_NAME,COUNTY_FIPS,UV_ Wh/m² (2000-2004),UV_ Wh/m² (2005-2009),UV_ Wh/m² (2010-2014),UV_ Wh/m² (2015-2019),UV_ Wh/m² (2020_2024)
390,District of Columbia,District of Columbia,11001,4286.810182,4233.748364,4291.252364,4154.570182,4235.43


For this row, we can use `.loc` to use the `COUNTY_FIPS` value to target the row value in the `STATE` column that should be filled in with a valid name:

In [414]:
melanoma_incidence.loc[melanoma_incidence['COUNTY_FIPS'] == '11001', 'STATE'] =  'District of Columbia'

In [415]:
melanoma_incidence.loc[2146]

COUNTY_NAME         District of Columbia(6)
STATE_NAME                              NaN
COUNTY_FIPS                           11001
RURAL_URBAN_NOTE                      Urban
AGE_RATE_PER100K                       10.0
AVG_ANNUAL_COUNT                       68.0
STATE                  District of Columbia
Name: 2146, dtype: object

For the second row with a NaN value, we will follow the same approach:

In [416]:
if uv_exposure[uv_exposure['COUNTY_FIPS'] == '72001' ].empty:
    print('FIPS CODE NOT FOUND')
else:
    print('UV INDEX MEASUREMENT FOUND FOR THIS FIPS CODE')


FIPS CODE NOT FOUND


Since there is no measurement for this county, we can drop this row from the `melanoma_incidence` dataset.

In [417]:
melanoma_incidence.dropna(subset=['STATE'], inplace=True)
melanoma_incidence.isna().sum()

COUNTY_NAME         0
STATE_NAME          1
COUNTY_FIPS         0
RURAL_URBAN_NOTE    0
AGE_RATE_PER100K    0
AVG_ANNUAL_COUNT    0
STATE               0
dtype: int64

In [418]:
df_insights(melanoma_incidence)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       7 │
├──────────────────┼─────────┤
│ Count of Rows    │       1 │
├──────────────────┼─────────┤
│ Missing Values   │       1 │
╘══════════════════╧═════════╛

 MISSING VALUES 

╒════════════╤════════════════════════════════╕
│ COLUMN     │   COUNT OF ROWS MISSING VALUES │
╞════════════╪════════════════════════════════╡
│ STATE_NAME │                              1 │
╘════════════╧════════════════════════════════╛

 DATATYPES 

╒══════════════════╤═════════╕
│ COLUMN           │ DTYPE   │
╞══════════════════╪═════════╡
│ COUNTY_NAME      │ str     │
├──────────────────┼─────────┤
│ STATE_NAME       │ str     │
├──────────────────┼─────────┤
│ COUNTY_FIPS      │ str     │
├──────────────────┼─────────┤
│ RURAL_URBAN_NOTE │ str     │
├──────────────────┼─────────┤
│ AGE_RATE_PER100K │ float64 │
├──────────────────┼─────────┤
│ AVG_ANNUAL_COUNT │ float64 │

In [419]:
melanoma_incidence.dtypes

COUNTY_NAME             str
STATE_NAME              str
COUNTY_FIPS             str
RURAL_URBAN_NOTE        str
AGE_RATE_PER100K    float64
AVG_ANNUAL_COUNT    float64
STATE                   str
dtype: object

# RURAL URBAN CONTINUUM CODES DATASET

In [420]:


continuum_codes = pd.read_excel('../datasets/rural-urban-continuum-codes-2023.xlsx',dtype={'FIPS': str})
continuum_codes.rename({'FIPS': 'COUNTY_FIPS', 
                        'State': 'STATE_ABB',
                        'County_Name': 'COUNTY_NAME',
                        'Population_2020': 'POPULATION_COUNT',
                        'Description': 'RUCC_DESCRIPTION'}, axis=1, inplace=True)
continuum_codes['COUNTY_FIPS'] = continuum_codes['COUNTY_FIPS'].str.zfill(5)
continuum_codes

,COUNTY_FIPS,STATE_ABB,COUNTY_NAME,POPULATION_COUNT,RUCC_2023,RUCC_DESCRIPTION
0,01001,AL,Autauga County,58805,2.0,"Metro - Counties in metro areas of 250,000 to ..."
1,01003,AL,Baldwin County,231767,3.0,Metro - Counties in metro areas of fewer than ...
2,01005,AL,Barbour County,25223,6.0,"Nonmetro - Urban population of 5,000 to 20,000..."
3,01007,AL,Bibb County,22293,1.0,Metro - Counties in metro areas of 1 million p...
4,01009,AL,Blount County,59134,1.0,Metro - Counties in metro areas of 1 million p...
...,...,...,...,...,...,...
3230,72151,PR,Yabucoa Municipio,30426,1.0,Metro - Counties in metro areas of 1 million p...
3231,72153,PR,Yauco Municipio,34172,2.0,"Metro - Counties in metro areas of 250,000 to ..."
3232,78010,VI,St. Croix Island,41004,5.0,"Nonmetro - Urban population of 20,000 or more,..."
3233,78020,VI,St. John Island,3881,9.0,"Nonmetro - Urban population of fewer than 5,00..."


In [421]:
continuum_codes.isna().sum()

COUNTY_FIPS         0
STATE_ABB           0
COUNTY_NAME         0
POPULATION_COUNT    0
RUCC_2023           2
RUCC_DESCRIPTION    0
dtype: int64

In [422]:
continuum_codes[continuum_codes['RUCC_2023'].isna()]

,COUNTY_FIPS,STATE_ABB,COUNTY_NAME,POPULATION_COUNT,RUCC_2023,RUCC_DESCRIPTION
3146,60030,AS,Rose Island,0,NaN,Not Applicable
3147,60040,AS,Swains Island,0,NaN,Not Applicable


These two rows have null values for Rural Urban Continuum Codes. <br>
Here, I will drop these two rows that are ireelevant for our analysis, since the population in these counties located in the American Samoa is zero.

In [423]:
continuum_codes.dropna(subset=['RUCC_2023'], inplace=True)
continuum_codes.isna().sum()

COUNTY_FIPS         0
STATE_ABB           0
COUNTY_NAME         0
POPULATION_COUNT    0
RUCC_2023           0
RUCC_DESCRIPTION    0
dtype: int64

In [424]:
df_insights(continuum_codes)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       6 │
├──────────────────┼─────────┤
│ Count of Rows    │    3233 │
├──────────────────┼─────────┤
│ Missing Values   │       0 │
╘══════════════════╧═════════╛

 DATATYPES 

╒══════════════════╤═════════╕
│ COLUMN           │ DTYPE   │
╞══════════════════╪═════════╡
│ COUNTY_FIPS      │ str     │
├──────────────────┼─────────┤
│ STATE_ABB        │ str     │
├──────────────────┼─────────┤
│ COUNTY_NAME      │ str     │
├──────────────────┼─────────┤
│ POPULATION_COUNT │ int64   │
├──────────────────┼─────────┤
│ RUCC_2023        │ float64 │
├──────────────────┼─────────┤
│ RUCC_DESCRIPTION │ str     │
╘══════════════════╧═════════╛


In [427]:
continuum_codes['RUCC_DESCRIPTION'].unique()

<StringArray>
[           'Metro - Counties in metro areas of 250,000 to 1 million population',
              'Metro - Counties in metro areas of fewer than 250,000 population',
      'Nonmetro - Urban population of 5,000 to 20,000, adjacent to a metro area',
               'Metro - Counties in metro areas of 1 million population or more',
     'Nonmetro - Urban population of fewer than 5,000, adjacent to a metro area',
 'Nonmetro - Urban population of fewer than 5,000, not adjacent to a metro area',
       'Nonmetro - Urban population of 20,000 or more, adjacent to a metro area',
  'Nonmetro - Urban population of 5,000 to 20,000, not adjacent to a metro area',
   'Nonmetro - Urban population of 20,000 or more, not adjacent to a metro area']
Length: 9, dtype: str

# SEER*STATS DATASET

In [430]:
seer_db = pd.read_csv('../datasets/seer-stats.csv')
seer_db.head(2)

,Rural-Urban Continuum Code,Age recode with single ages and 90+,Sex,Year of diagnosis,"Race and origin recode (NHW, NHB, NHAIAN, NHAPI, Hispanic)",Site recode ICD-O-3 2023 Revision Expanded,"ICD-O-3 Hist/behav, malignant",Behavior code ICD-O-3,Combined Summary Stage with Expanded Regional Codes (2004+)
0,Counties in metropolitan areas ge 1 million pop,70 years,Male,2021,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Localized only
1,Counties in metropolitan areas ge 1 million pop,61 years,Female,2017,Non-Hispanic White,Melanoma Of The Skin,"8720/3: Malignant melanoma, NOS",Malignant,Localized only


In [426]:
df_insights(seer_db)


 DF SUMMARY 

╒══════════════════╤═════════╕
│ Parameter        │   Value │
╞══════════════════╪═════════╡
│ Count of Columns │       9 │
├──────────────────┼─────────┤
│ Count of Rows    │  156857 │
├──────────────────┼─────────┤
│ Missing Values   │       0 │
╘══════════════════╧═════════╛

 DATATYPES 

╒═════════════════════════════════════════════════════════════╤═════════╕
│ COLUMN                                                      │ DTYPE   │
╞═════════════════════════════════════════════════════════════╪═════════╡
│ Rural-Urban Continuum Code                                  │ str     │
├─────────────────────────────────────────────────────────────┼─────────┤
│ Age recode with single ages and 90+                         │ str     │
├─────────────────────────────────────────────────────────────┼─────────┤
│ Sex                                                         │ str     │
├─────────────────────────────────────────────────────────────┼─────────┤
│ Year of diagnosis       